In [1]:
import pandas as pd
import numpy as np
import os
import sys
import mne

In [2]:
grass_dir = '//mad3/MGH-NEURO-CDAC/Datasets_ConvertedData/sleeplab/grass_data/'

In [3]:
grass_folders = os.listdir(grass_dir)

In [4]:
grass_folders.remove('.DS_Store')
grass_folders.remove('____DO_NOT_USE____')
grass_folders.remove('____DUPLICATES____')
grass_folders.remove('____INCOMPLETE____')
grass_folders.remove('____UNMATCHED____')

In [5]:
grass_df = pd.DataFrame(columns=['fname', 'signals_avail', 'labels_avail', 'annotations_avail', 'recording_added'])

In [6]:
grass_df['fname'] = grass_folders
grass_df['signals_avail'] = 0
grass_df['signals_name'] = ''
grass_df['labels_avail'] = 0
grass_df['annotations_avail'] = 0
grass_df['start_datetime'] = np.nan
grass_df['recording_added'] = 0

In [12]:
for jloc, row in grass_df.iterrows():
    
    start_time = np.nan
    name = row.fname
    if name in ['____DO_NOT_USE____', '____DUPLICATES____', '____INCOMPLETE____', '____UNMATCHED____']:
        continue
    
    files = os.listdir(os.path.join(grass_dir, name))
    signal_file = [x for x in files if all(['.mat' in x.lower(), 'signal' in x.lower()])]
    label_file = [x for x in files if all(['.mat' in x.lower(), 'label' in x.lower()])]
    annotation = [x for x in files if all(['.csv' in x.lower(), 'annotation' in x.lower()])]
    edf_file = [x for x in files if '.edf' in x]

    if len(signal_file) > 0:
        grass_df.loc[jloc, 'signals_avail'] = 1
        
        if len(signal_file) > 1:    
            if len(label_file) > 0:
                labelfilename = label_file[0].replace('Labels_', '').replace('.mat', '')
                signal_file = signal_file = [x for x in signal_file if labelfilename in x]
                assert len(signal_file) == 1
            elif len([x for x in signal_file if 'Exported' in x]) > 0:
                signal_file = [x for x in signal_file if 'Exported' in x]
                assert len(signal_file) == 1
        
        grass_df.loc[jloc, 'signals_name'] = signal_file[0]
            
    if len(label_file) > 0:
        grass_df.loc[jloc, 'labels_avail'] = 1
    if len(annotation) > 0:
        grass_df.loc[jloc, 'annotations_avail'] = 1
        
        
    if len(edf_file) > 0:
        edf = mne.io.read_raw_edf(os.path.join(grass_dir, name, edf_file[0]), stim_channel=None, preload=False, verbose=False)
        start_time = edf.info['meas_date'].replace(tzinfo=None)
    elif len(annotation) > 0:
        annot = pd.read_csv(os.path.join(grass_dir, name, annotation[0]))
        start_recording = annot.loc[annot.event == 'Start Recording']
        if len(start_recording) == 1:
            
            if os.path.exists(os.path.join(grass_dir, name, 'ID.csv')):
                study_id = pd.read_csv(os.path.join(grass_dir, name, 'ID.csv'))
                start_time = pd.Timestamp(study_id.DateOfVisit.values[0] + ' ' + start_recording.time.values[0])
                
            elif os.path.exists(os.path.join(grass_dir, name, 'Study.csv')):
                study = pd.read_csv(os.path.join(grass_dir, name, 'Study.csv'))
                start_time = pd.Timestamp(study.loc[study.Variable == 'Fld_StartDateTime'].Data.values[0])
                assert pd.Timestamp(start_recording.time.values[0]).time() == start_time.time()
            else:
                print('break No ID or Study.csv available')
                break
    else:
#         print('No EDF or annotation.csv available')
#         print(files)
        continue
        
    grass_df.loc[jloc, 'start_datetime'] = str(start_time)
    
grass_df['recording_to_add'] = 0 
grass_df['recording_to_add'] = np.all([grass_df['signals_avail'] == 1, pd.notna(grass_df['recording_to_add'])], axis=0).astype(int)  

No EDF or annotation.csv available
['02_22_17_2948495_pre - Copy.docm', '02_22_17_2948495_pre.docm', 'ID.csv', 'ID_archived.csv', 'Kaskan_Madeline_022217_2237.000.dat', 'KASKAN_MADELINE_022217_2237.000.Diagnostic MD 042209_R2.doc', 'Kaskan_Madeline_022217_2237.000.lay', 'KASKAN_MADELINE_022217_2237.000.MGH PSG LONG.doc', 'KASKAN_MADELINE_022217_2237.000.XML PSG Statistics.doc', 'Kaskan_Madeline_022217_2237.000__02_22_17_2948495_pre (2).csv', 'Kaskan_Madeline_022217_2237.000__02_22_17_2948495_pre (2).docm', 'KASKAN_MADELINE_022217_2237.xml', 'lay_metadata.csv', 'MedicalReport.csv', 'Observation.csv', 'Observation.xml', 'raw', 'Signal_Kaskan_Madeline_022217_2237.000.mat', 'SleepStatistics.csv', 'SleepStatistics.xml', 'Study.csv', 'Study.xml', 'Tech report_Kaskan.docx', '~$ch report_Kaskan.docx', '~$_22_17_2948495_pre.docm']
No EDF or annotation.csv available
['10_08_16_ 1738523_pre.docm', 'ID.csv', 'ID_archived.csv', 'lay_metadata.csv', 'MedicalReport.csv', 'Observation.csv', 'Observatio

<ipython-input-12-1ebe020e3cc1>:35: RuntimeWarning: Channel names are not unique, found duplicates for: {'Leak'}. Applying running numbers for duplicates.
  edf = mne.io.read_raw_edf(os.path.join(grass_dir, name, edf_file[0]), stim_channel=None, preload=False, verbose=False)
<ipython-input-12-1ebe020e3cc1>:35: RuntimeWarning: Channel names are not unique, found duplicates for: {'Leak'}. Applying running numbers for duplicates.
  edf = mne.io.read_raw_edf(os.path.join(grass_dir, name, edf_file[0]), stim_channel=None, preload=False, verbose=False)
<ipython-input-12-1ebe020e3cc1>:35: RuntimeWarning: Number of records from the header does not match the file size (perhaps the recording was not stopped before exiting). Inferring from the file size.
  edf = mne.io.read_raw_edf(os.path.join(grass_dir, name, edf_file[0]), stim_channel=None, preload=False, verbose=False)


No EDF or annotation.csv available
['Costello_Maureen_041017_2302.000.dat', 'Costello_Maureen_041017_2302.000.lay', 'COSTELLO_MAUREEN_041017_2302.xml', 'ID.csv', 'ID_archived.csv', 'lay_metadata.csv', 'MedicalReport.csv', 'Observation.csv', 'Observation.xml', 'raw', 'Signal_Costello_Maureen_041017_2302.000.mat', 'SleepStatistics.csv', 'SleepStatistics.xml', 'Study.csv', 'Study.xml']


<ipython-input-12-1ebe020e3cc1>:35: RuntimeWarning: Channel names are not unique, found duplicates for: {'C3-M2'}. Applying running numbers for duplicates.
  edf = mne.io.read_raw_edf(os.path.join(grass_dir, name, edf_file[0]), stim_channel=None, preload=False, verbose=False)


No EDF or annotation.csv available
['ID.csv', 'ID_archived.csv', 'raw', 'Study.csv', 'Study.xml']
No EDF or annotation.csv available
['05_24_16_1796350_pre.doc.docm', 'Arroyo tech report 052416.docx', 'Arroyo_Gladys I_052416_2203.000.dat', 'Arroyo_Gladys I_052416_2203.000.lay', 'Arroyo_Gladys I_052416_2203.000__05_24_16_1796350_pre.csv', 'Arroyo_Gladys I_052416_2203.000__05_24_16_1796350_pre.doc.docm', 'ARROYO_GLADYS I_052416_2203.xml', 'hx note.docx', 'ID.csv', 'ID_archived.csv', 'lay_metadata.csv', 'MedicalReport.csv', 'Observation.csv', 'Observation.xml', 'PSG 1.docx', 'raw', 'Req.docx', 'Signal_Arroyo_Gladys I_052416_2203.000.mat', 'Study.csv', 'Study.xml']
No EDF or annotation.csv available
['05_04_16_3745814_pre.docm', 'hx note.docx', 'ID.csv', 'ID_archived.csv', 'lay_metadata.csv', 'LONCZAK_JONATHAN_050416_2243.000.CPAP ALl Night MD 022209_R2.doc', 'Lonczak_Jonathan_050416_2243.000.dat', 'Lonczak_Jonathan_050416_2243.000.lay', 'LONCZAK_JONATHAN_050416_2243.000.MGH CPAP LONG.doc'

<ipython-input-12-1ebe020e3cc1>:35: RuntimeWarning: Channel names are not unique, found duplicates for: {'Leak'}. Applying running numbers for duplicates.
  edf = mne.io.read_raw_edf(os.path.join(grass_dir, name, edf_file[0]), stim_channel=None, preload=False, verbose=False)


No EDF or annotation.csv available
['Dando_Jeffrey_071208_2306.000.dat', 'DANDO_JEFFREY_071208_2306.000.JME_CPAP_Test.doc', 'DANDO_JEFFREY_071208_2306.000.JME_PSG_Test.doc', 'Dando_Jeffrey_071208_2306.000.lay', 'DANDO_JEFFREY_071208_2306.000.MGH PSG LONG.doc', 'DANDO_JEFFREY_071208_2306.000.XML PSG Statistics.doc', 'DANDO_JEFFREY_071208_2306.xml', 'ID.csv', 'ID_archived.csv', 'lay_metadata.csv', 'MedicalReport.csv', 'raw', 'Signal_Dando_Jeffrey_071208_2306.000.mat', 'SleepStatistics.csv', 'SleepStatistics.xml']


In [15]:
grass_df.to_csv('grass_files_start_datetime.csv', index=False)

In [16]:
grass_df.loc[grass_df.recording_to_add == 1].to_csv('grass_files_start_datetime_to_add.csv', index=False)

In [17]:
grass_df.loc[grass_df.recording_to_add == 1]

,fname,signals_avail,labels_avail,annotations_avail,recording_added,signals_name,start_datetime,recording_to_add
0,Anatasia_Lori_091611_2209.000,1,0,1,0,Signal_TwinData5_Exported_34.mat,2011-09-16 22:09:21,1
1,Gronewold_Catherine W_041615_2224.000,1,1,1,0,Signal_Gronewold_Catherine W_041615_2224.000.mat,2015-04-16 22:23:40,1
2,Morris_Martha_112512_2118.000,1,0,1,0,Signal_TwinData6_Exported_759.mat,2012-11-25 21:18:20,1
3,Schlein_Toby_012717_2300.000,1,1,1,0,Signal_Schlein_Toby_012717_2300.000.mat,2017-01-27 22:59:57,1
4,Callahan_Marian_042010_2158.000,1,0,1,0,Signal_TwinData3_Exported_285.mat,2010-04-20 21:57:34,1
...,...,...,...,...,...,...,...,...
21309,Monahan_Joan_102715_2217.000,1,0,1,0,Signal_TwinData13_Exported_607.mat,2015-10-27 22:17:08,1
21310,Carpenito_Cody_083112_2138.000,1,0,1,0,Signal_TwinData6_Exported_151.mat,2012-08-31 21:38:13,1
21311,Scouras_Peter_092813_2202.000,1,1,1,0,Signal_Scouras_Peter_092813_2202.000.mat,2013-09-28 22:02:19,1
21312,Rahman_Husneara_061013_2121.000,1,1,1,0,Signal_Rahman_Husneara_061013_2121.000.mat,2013-06-10 21:21:30,1
